In [ ]:
import os
import argparse
from langchain.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

def load_documents(data_folder: str):
    """Load all .txt and .pdf files from the given folder."""
    txt_loader = DirectoryLoader(data_folder, glob="**/*.txt", loader_cls=TextLoader)
    pdf_loader = DirectoryLoader(data_folder, glob="**/*.pdf", loader_cls=PyPDFLoader)
    docs = txt_loader.load() + pdf_loader.load()
    print(f"Loaded {len(docs)} documents")
    return docs

def chunk_documents(docs, chunk_size=500, chunk_overlap=100):
    """Split documents into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(docs)
    print(f"Created {len(chunks)} chunks")
    return chunks

def build_vectorstore(chunks, persist_dir: str, embedding_model: str):
    """Create and persist a Chroma vector store."""
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model)
    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_dir
    )
    vectordb.persist()
    print(f"Vector store saved to {persist_dir}")
    return vectordb

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Ingest documents and build vector store.")
    parser.add_argument("--data_folder", type=str, default="./data", help="Folder containing documents")
    parser.add_argument("--persist_dir", type=str, default="./med_chroma", help="Directory to save vector store")
    parser.add_argument("--embedding_model", type=str, default="BAAI/bge-small-en-v1.5", help="HuggingFace embedding model")
    parser.add_argument("--chunk_size", type=int, default=500)
    parser.add_argument("--chunk_overlap", type=int, default=100)
    args = parser.parse_args()

    docs = load_documents(args.data_folder)
    chunks = chunk_documents(docs, args.chunk_size, args.chunk_overlap)
    build_vectorstore(chunks, args.persist_dir, args.embedding_model)